# 2.1 Pandas & Polars — Tabular Data Processing> The two dominant Python dataframe libraries. Pandas is ubiquitous; Polars is the fast newcomer.---

## Pandas Fundamentals

In [ ]:
import pandas as pdimport numpy as np# Create a DataFrame — like a database tabledf = pd.DataFrame({    "patient_id": ["P001", "P002", "P003", "P004", "P005"],    "age": [45, 62, 38, 71, 55],    "diagnosis": ["diabetes", "hypertension", "diabetes", "copd", "hypertension"],    "lab_value": [6.5, 140.0, 7.1, np.nan, 135.0],    "admission_date": pd.to_datetime(["2024-01-15", "2024-01-16", "2024-02-01", "2024-02-10", "2024-03-05"]),})print(df)print(f"\nShape: {df.shape}")print(f"\nDtypes:\n{df.dtypes}")

In [ ]:
# Filtering, selecting, transforming# Like SQL: SELECT * FROM patients WHERE age > 50 AND diagnosis = 'hypertension'filtered = df[(df["age"] > 50) & (df["diagnosis"] == "hypertension")]print("Filtered:\n", filtered)# GroupBy — like SQL GROUP BYstats = df.groupby("diagnosis").agg(    count=("patient_id", "count"),    avg_age=("age", "mean"),    avg_lab=("lab_value", "mean"),).round(1)print("\nGrouped stats:\n", stats)

In [ ]:
# Handling missing data — critical for healthcare dataprint("Missing values:\n", df.isnull().sum())# Fill, drop, or flagdf["lab_value_filled"] = df["lab_value"].fillna(df["lab_value"].median())df["has_lab"] = df["lab_value"].notna()print(df[["patient_id", "lab_value", "lab_value_filled", "has_lab"]])

## Polars — The Fast AlternativePolars is Rust-based, uses lazy evaluation (like Spark!), and is 10-100x faster than Pandas for large datasets.

In [ ]:
import polars as pl# Same data in Polarsdf_pl = pl.DataFrame({    "patient_id": ["P001", "P002", "P003", "P004", "P005"],    "age": [45, 62, 38, 71, 55],    "diagnosis": ["diabetes", "hypertension", "diabetes", "copd", "hypertension"],    "lab_value": [6.5, 140.0, 7.1, None, 135.0],})# Polars uses expressions — chainable, like Sparkresult = (    df_pl    .filter(pl.col("age") > 50)    .with_columns(        pl.col("lab_value").fill_null(pl.col("lab_value").median()).alias("lab_filled"),        (pl.col("age") * 365).alias("age_days"),    )    .select(["patient_id", "diagnosis", "lab_filled", "age_days"]))print(result)

In [ ]:
# Lazy evaluation — like Spark's transformations vs actionslazy_result = (    df_pl.lazy()    .filter(pl.col("age") > 40)    .group_by("diagnosis")    .agg([        pl.count().alias("count"),        pl.col("age").mean().alias("avg_age"),    ])    .sort("count", descending=True)    .collect()  # triggers execution — like Spark .collect())print(lazy_result)

## 🏋️ Exercise: Clean & Aggregate Patient DataGiven a messy dataset, clean it and produce summary statistics per diagnosis.

In [ ]:
# Exercise dataimport pandas as pdimport numpy as npraw = pd.DataFrame({    "id": range(1, 101),    "age": np.random.randint(18, 90, 100),    "diagnosis": np.random.choice(["diabetes", "copd", "heart_failure", None], 100),    "cost": np.random.uniform(100, 50000, 100).round(2),})raw.loc[np.random.choice(100, 10, replace=False), "cost"] = np.nan# TODO:# 1. Drop rows where diagnosis is null# 2. Fill missing cost with median cost per diagnosis# 3. Group by diagnosis and compute: count, mean cost, max cost# 4. Sort by mean cost descendingprint(raw.head(10))